# Real-Time Credit Scoring — Standard Table/View + REST API (Showcase)

Notebook ini men-*showcase* alur **real-time decision engine** CLIK dengan lookup fitur dari **VIEW di atas STANDARD TABLE** (`SUBJECT_FEATURES_ENCODED`), dijalankan **di dalam Snowflake Notebook**:

1. **Step 3 — Lookup fitur dari View/Standard Table** (`SUBJECT_FEATURES_ENCODED`) untuk ambil 60 fitur ter-encode.
2. **Step 4 — Call Model Service via REST API** (SPCS ingress) dari dalam notebook via **External Access Integration**.
3. **Step 5 — Mapping bisnis**: PD → credit score → decision.

> Bandingkan dengan **`04b_realtime_hybrid_rest.ipynb`** yang lookup ke **Hybrid Table** (`SUBJECT_FEATURES_ENCODED_HT`).
> - View/standard table → setiap lookup men-**scan** micro-partition (bytes_scanned besar).
> - Hybrid table → **point lookup by PK** (bytes_scanned ~0), ideal untuk real-time 100M baris & konkurensi tinggi.

### Prasyarat (WAJIB)
- Service `CLIK_PD_SERVICE` sudah **READY** (lihat `04b_deploy_service.py`).
- View `SUBJECT_FEATURES_ENCODED` sudah ada (lihat `04a_batch_scoring.sql`).
- External Access Integration **`CLIK_SPCS_EAI`** (network rule egress) sudah dibuat (lihat `04b_notebook_rest_setup.sql`). Secret **tidak diperlukan** — tiap peserta isi PAT sendiri di Cell 1.
- **Attach EAI ke notebook ini:** menu `⋯` → **Notebook settings** → **External access integrations** → aktifkan `CLIK_SPCS_EAI` → restart.
- **Isi PAT** di Cell 1 (`PAT_TOKEN = "..."`) dengan PAT milik Anda sendiri.

In [ ]:
# Cell 1 - Setup: session, ingress URL, PAT dari secret
import json
import time
import requests
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("ALTER SESSION SET USE_CACHED_RESULT = FALSE").collect()

# Ingress URL diambil dinamis dari service (tanpa hardcode)
ep = session.sql("SHOW ENDPOINTS IN SERVICE CLIK_WORKSHOP2.PUBLIC.CLIK_PD_SERVICE").collect()
INGRESS_URL = ep[0]["ingress_url"]
ENDPOINT_URL = f"https://{INGRESS_URL}/predict-proba"

# >>> ISI PAT ANDA DI SINI (tiap peserta HOL pakai PAT masing-masing) <<<
# Buat PAT: Snowsight -> profil (kiri bawah) -> Settings -> Authentication ->
# Programmatic access tokens -> Generate new token. Tempel di bawah:
PAT_TOKEN = "PASTE_YOUR_PAT_HERE"
HEADERS = {
    "Authorization": f'Snowflake Token="{PAT_TOKEN}"',
    "Content-Type": "application/json",
}
print("Endpoint:", ENDPOINT_URL)

## Step 3 — Lookup fitur dari View / Standard Table

Ambil 60 fitur ter-encode dari **`SUBJECT_FEATURES_ENCODED`** (view di atas standard table `SUBJECT_FEATURES`). Encoding (OHE + DTI_RATIO) dihitung on-the-fly.

In [ ]:
# Cell 2 - Step 3: lookup fitur dari VIEW / standard table
SUBJECT_IDS = ['SUBJ000000020', 'SUBJ000000044', 'SUBJ000000068']
placeholders = ", ".join([f"'{s}'" for s in SUBJECT_IDS])

t0 = time.perf_counter()
df = session.sql(f"""
    SELECT * EXCLUDE (SUBJECT_ID)
    FROM CLIK_WORKSHOP2.PUBLIC.SUBJECT_FEATURES_ENCODED
    WHERE SUBJECT_ID IN ({placeholders})
""").to_pandas()
lookup_ms = (time.perf_counter() - t0) * 1000.0
print(f"[View/Standard Lookup] {df.shape[0]} baris x {df.shape[1]} kolom dalam {lookup_ms:.0f} ms (scan micro-partition)")
df.head()

## Step 4 — Call Model Service via REST API

POST fitur ke endpoint SPCS. Payload `dataframe_split` **wajib** menyertakan key `index` → gunakan `df.to_json(orient="split")` (jangan `index=False`).

In [ ]:
# Cell 3 - Step 4: call Model Service via REST API (dari dalam notebook, lewat EAI)
split_obj = json.loads(df.to_json(orient="split"))
payload = {"dataframe_split": split_obj}

t0 = time.perf_counter()
resp = requests.post(ENDPOINT_URL, headers=HEADERS, json=payload, timeout=30)
infer_ms = (time.perf_counter() - t0) * 1000.0
print(f"[Model Service REST] HTTP {resp.status_code} dalam {infer_ms:.0f} ms")
resp.raise_for_status()
result = resp.json()
print(json.dumps(result, indent=2)[:600])

## Step 5 — Mapping Bisnis (PD → Credit Score → Decision)

Output service: `{"data": [[idx, {..., PREDICT_PROBA_1}]]}`. `PREDICT_PROBA_1` = P(default) = PD score.

In [ ]:
# Cell 4 - Step 5: mapping bisnis
import pandas as pd

def extract_pd(row_out):
    obj = row_out[1] if isinstance(row_out, list) else row_out
    return float(obj["PREDICT_PROBA_1"]) if isinstance(obj, dict) else float(row_out[-1])

rows = []
for sid, row_out in zip(SUBJECT_IDS, result["data"]):
    pd_score = extract_pd(row_out)
    credit_score = 300 + round(550 * (1 - pd_score))
    decision = "APPROVE" if pd_score < 0.10 else ("REVIEW" if pd_score < 0.30 else "REJECT")
    rows.append({"SUBJECT_ID": sid, "PD": round(pd_score, 4), "CREDIT_SCORE": credit_score, "DECISION": decision})

result_df = pd.DataFrame(rows)
print(f"Total latency (lookup + REST): {lookup_ms + infer_ms:.0f} ms")
result_df